# Reasoning Traces Analysis

View reasoning traces and their token lengths using the Qwen tokenizer.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

In [ ]:
# Load parsed results
data_dir = Path("data/reasoning_traces")
parsed_results_file = data_dir / "parsed_results.json"

with open(parsed_results_file, "r") as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f"Loaded {len(df)} samples")
print(f"Conditions: {df['condition'].value_counts().to_dict()}")

In [ ]:
# Load Qwen tokenizer and calculate token counts for reasoning traces
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B")

df["reasoning_tokens"] = df["reasoning"].apply(
    lambda x: len(tokenizer.encode(x)) if isinstance(x, str) else 0
)

for condition in df["condition"].unique():
    subset = df[df["condition"] == condition]
    print(f"\n{condition} - Reasoning token count stats:")
    print(f"  Mean: {subset['reasoning_tokens'].mean():.0f}")
    print(f"  Min:  {subset['reasoning_tokens'].min()}")
    print(f"  Max:  {subset['reasoning_tokens'].max()}")
    print(f"  Median: {subset['reasoning_tokens'].median():.0f}")

In [ ]:
# Display responses grouped by sample, showing both conditions side by side
without_df = df[df["condition"] == "without_intent"].reset_index(drop=True)
with_df = df[df["condition"] == "with_intent"].reset_index(drop=True)

for idx in range(len(without_df)):
    wo = without_df.iloc[idx]
    wi = with_df.iloc[idx]
    
    print(f"\n{'='*80}")
    print(f"Sample {idx + 1} (Wildguard ID: {wo['wildguard_id']})")
    print(f"{'='*80}")
    
    print(f"\nPrompt:\n  {wo['prompt'][:200]}{'...' if len(wo['prompt']) > 200 else ''}")
    print(f"\nIntent:\n  {wo['intent']}")
    print(f"\nResponse:\n  {wo['response'][:200]}{'...' if len(wo['response']) > 200 else ''}")
    
    print(f"\n--- Without Intent ({wo['reasoning_tokens']} tokens) ---")
    print(f"  Predicted: {wo['predicted']}")
    
    print(f"\n--- With Intent ({wi['reasoning_tokens']} tokens) ---")
    print(f"  Predicted: {wi['predicted']}")
    
    print(f"\n  Ground truth: {wo['ground_truth']}")